# Smoke Test, is my runtime healthy?

Run this **first** on day one. It imports every package the course uses, prints each version,
runs one trivial operation per library, and prints a green **all systems go** or a specific
failure pointing at the cheat sheet. Its printed output is also the course's "tested as of"
record, copy the versions into `README.md`.

In [ ]:
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# Install the course's extra packages (safe to re-run).
%pip install -q -r requirements.txt -c constraints.txt

In [ ]:
import importlib, platform
print("Python", platform.python_version())

results = []
def check(mod, op):
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        op(m)
        results.append((mod, ver, "ok", ""))
    except Exception as e:
        results.append((mod, "-", "FAIL", f"{type(e).__name__}: {e}"))

# Boring Colab-preinstalled stack
check("numpy",        lambda m: m.array([1,2,3]).mean())
check("pandas",       lambda m: m.DataFrame({"a":[1,2]}).sum())
check("sklearn",      lambda m: __import__("sklearn.linear_model", fromlist=["LogisticRegression"]))
check("matplotlib",   lambda m: __import__("matplotlib.pyplot", fromlist=["plot"]))
check("PIL",          lambda m: __import__("PIL.Image", fromlist=["new"]).new("RGB",(4,4)))
check("requests",     lambda m: hasattr(m, "get"))
check("bs4",          lambda m: m.BeautifulSoup("<b>hi</b>", "html.parser").text)

# The pinned extras
check("sentence_transformers", lambda m: hasattr(m, "SentenceTransformer"))
check("google.generativeai",   lambda m: hasattr(m, "GenerativeModel"))

print(f"{'package':24s} {'version':12s} status")
for mod, ver, status, msg in results:
    print(f"{mod:24s} {ver:12s} {status}  {msg}")

fails = [r for r in results if r[2] != "ok"]
print()
if not fails:
    print("ALL SYSTEMS GO. Your runtime is healthy.")
else:
    print("Some checks FAILED. For each, see ../kits/common-errors-cheatsheet.md (entry 5,")
    print("ModuleNotFoundError). Re-run the install cell, then re-run this one.")
    for mod, *_ in fails:
        print("  - missing or broken:", mod)

In [ ]:
# One end-to-end micro-op per core tool (skips gracefully if a package is missing).
try:
    from sklearn.linear_model import LogisticRegression
    import numpy as np
    LogisticRegression().fit(np.array([[0],[1],[2],[3]]), [0,0,1,1])
    print("logistic regression: fit ok (Week 3)")
except Exception as e:
    print("logistic regression skipped:", e)

try:
    from sentence_transformers import SentenceTransformer
    v = SentenceTransformer("all-MiniLM-L6-v2").encode(["hello world"])
    print("sentence-transformers: embedded one sentence ->", v.shape, "(Week 5)")
except Exception as e:
    print("sentence-transformers skipped (install + network needed):", type(e).__name__)

import os
if os.environ.get("GEMINI_API_KEY"):
    try:
        import google.generativeai as genai
        genai.configure(api_key=os.environ["GEMINI_API_KEY"])
        r = genai.GenerativeModel("gemini-2.5-flash").generate_content("say ok")
        print("gemini: replied (Week 7)")
    except Exception as e:
        print("gemini call skipped:", type(e).__name__)
else:
    print("gemini: no GEMINI_API_KEY set, Week 7 will use its recorded sample (fine for now)")